In [ ]:
#CELL 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


import os
PROJECT_DIR = '/content/drive/MyDrive/MusicGenerationCSE425'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project directory: {PROJECT_DIR}")

Mounted at /content/drive
Project directory: /content/drive/MyDrive/MusicGenerationCSE425


In [ ]:
#CELL 1: Setup
!pip install pretty_midi -q
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd, pretty_midi
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/MusicGenerationCSE425'
MAESTRO_DIR  = f'{PROJECT_DIR}/data/raw_midi/maestro-v3.0.0'
MIDI_OUT_DIR = f'{PROJECT_DIR}/outputs/generated_midis'
os.makedirs(MIDI_OUT_DIR, exist_ok=True)

df = pd.read_csv(f"{MAESTRO_DIR}/maestro-v3.0.0.csv")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 74.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.6 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELL 2: Baseline 1 — Random Note Generator


def generate_random_midi(output_path, n_notes=80, duration=30.0):
    """Generate a MIDI file by placing random notes uniformly. This is the trivial baseline — no musical knowledge used.
       Args: output_path: where to save the .mid file, n_notes: number of random notes to generate, duration : total duration in seconds"""
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)

    current_time = 0.0
    for _ in range(n_notes):
        pitch = np.random.randint(21, 109) #Random piano key
        velocity = np.random.randint(40, 100) #Random velocity
        dur = np.random.choice([0.25, 0.5, 1.0]) #Random duration
        note = pretty_midi.Note(velocity=velocity, pitch=pitch, start=current_time, end=current_time + dur)
        piano.notes.append(note)
        current_time += dur
        if current_time > duration:
            break

    midi.instruments.append(piano)
    midi.write(output_path)
    return output_path

#Generate 5 random samples
print("Generating Random Baseline samples...")
for i in range(5):
    path = f"{MIDI_OUT_DIR}/baseline_random_{i+1}.mid"
    generate_random_midi(path)
    print(f"  Saved: {path}")
print("Random baseline samples generated!")

Generating Random Baseline samples...
  Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_random_1.mid
  Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_random_2.mid
  Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_random_3.mid
  Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_random_4.mid
  Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_random_5.mid
Random baseline samples generated!


In [ ]:
#CELL 3: Baseline 2 — Markov Chain Music Model
#Captures local melodic patterns
from collections import defaultdict

def build_markov_transition_matrix(df_train, maestro_dir, n_files=100):
    """
    Build a first-order pitch transition matrix from training MIDI files.
    transition_matrix[i][j] = P(next pitch = j | current pitch = i)
    Args: df_train: training split DataFrame, maestro_dir: path to MAESTRO root folder, n_files: max files to process (use more for better quality)
    Returns: 88x88 numpy probability matrix empirical note duration distribution (list of floats)
    """

    counts = np.zeros((88, 88), dtype=np.float64)
    durations = []
    sample_files = df_train['midi_filename'].sample(min(n_files, len(df_train)), random_state=42).tolist()

    print(f"Building Markov model from {len(sample_files)} files...")
    for fname in sample_files:
        fpath = os.path.join(maestro_dir, fname)
        try:
            midi = pretty_midi.PrettyMIDI(fpath)
        except Exception:
            continue

        for inst in midi.instruments:
            #Sort notes by onset time
            notes = sorted(inst.notes, key=lambda n: n.start)
            pitches = [n.pitch for n in notes if 21 <= n.pitch <= 108]
            durs = [n.end - n.start for n in notes if 21 <= n.pitch <= 108]

            #Count transitions
            for i in range(len(pitches) - 1):
                p_curr = pitches[i] - 21
                p_next = pitches[i+1] - 21
                counts[p_curr, p_next] += 1
            durations.extend(durs)

    #Normalize rows to get probabilities
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    transition_matrix = counts / row_sums
    print(f"Markov matrix built. Duration samples: {len(durations)}")
    return transition_matrix, durations

def generate_markov_midi(transition_matrix, durations, output_path, n_notes=100, start_pitch=60):
    """
    Generate a MIDI file using the Markov chain transition matrix.
    Args:
    transition_matrix: 88x88 probability matrix, durations: list of real note durations for sampling, output_path: output .mid file path, n_notes: number of notes to generate, start_pitch: MIDI number for the first note (default: middle C)
    """
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0) #Acoustic Grand Piano

    current_pitch = start_pitch - 21
    current_time = 0.0

    for _ in range(n_notes):

        probs = transition_matrix[current_pitch]
        if probs.sum() == 0:
            probs = np.ones(88) / 88
        next_pitch_idx = np.random.choice(88, p=probs)

        #Sample duration from empirical distribution
        dur = np.random.choice(durations) if durations else 0.5
        dur = np.clip(dur, 0.1, 2.0)

        #Create note
        note = pretty_midi.Note(velocity=70, pitch = next_pitch_idx + 21, start = current_time, end = current_time + dur) #For pitch denormalize
        piano.notes.append(note)
        current_time += dur
        current_pitch = next_pitch_idx

    midi.instruments.append(piano)
    midi.write(output_path)
    return output_path


#Build Markov model from training data
train_df = df[df['split'] == 'train']
T_matrix, dur_list = build_markov_transition_matrix(train_df, MAESTRO_DIR, n_files=150)

#Save model for reuse
np.save(f"{PROJECT_DIR}/models_saved/markov_transition_matrix.npy", T_matrix)
np.save(f"{PROJECT_DIR}/models_saved/markov_durations.npy", np.array(dur_list))
print("Markov model saved!")

#Generate 5 Markov samples
print("\nGenerating Markov Chain samples...")
for i in range(5):
    path = f"{MIDI_OUT_DIR}/baseline_markov_{i+1}.mid"
    generate_markov_midi(T_matrix, dur_list, path)
    print(f"Saved: {path}")

print("Markov baseline samples generated!")

Building Markov model from 150 files...
Markov matrix built. Duration samples: 863702
Markov model saved!

Generating Markov Chain samples...
Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_markov_1.mid
Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_markov_2.mid
Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_markov_3.mid
Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_markov_4.mid
Saved: /content/drive/MyDrive/MusicGenerationCSE425/outputs/generated_midis/baseline_markov_5.mid
Markov baseline samples generated!
